In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from train_medsos import buat_pipeline, cv_group
from fitur import normalisasi, ciri_url

df = pd.read_csv("data/dataset_medsos.csv")
game = pd.read_csv("data/game_items.csv")
print(df.shape, game.shape)
df.sample(8, random_state=1)[["teks", "label", "kategori"]]

In [ ]:
# Komposisi data
print(df["label"].map({0: "normal", 1: "phishing"}).value_counts(), "\n")
print(pd.crosstab(df["kategori"], df["label"]))
print("\nJumlah template (grup):", df["grup"].nunique())

## Seperti apa teks setelah dinormalisasi?
Tautan diganti `zurl` dan ciri-cirinya ditambahkan di akhir sebagai token `z...`. Pesan tanpa tautan mendapat `ztanpalink`.

In [ ]:
contoh = [
    "Klik http://dana-hadiah.xyz/klaim untuk ambil saldo gratis Rp1.000.000!!",
    "Seminar gratis hari Sabtu, daftar: https://www.kampusku.ac.id/seminar",
    "Kak tolong kirim kode OTP 482913 yang masuk ke HP kamu",
]
for t in contoh:
    print(t, "\n  ->", normalisasi(t), "\n")

## Evaluasi 1: GroupKFold per template
Varian dari template yang sama tidak boleh muncul di data latih dan validasi sekaligus, kalau tidak skornya menipu (model cuma menghafal template). Parameter `C` dipilih dari sini.

In [ ]:
hasil = []
for C in (0.3, 1, 3, 10, 30):
    acc, f1 = cv_group(df, C, char=False)
    hasil.append({"C": C, "akurasi": round(acc, 3), "F1_phishing": round(f1, 3)})
pd.DataFrame(hasil)

In [ ]:
C_TERPILIH = max(hasil, key=lambda r: r["F1_phishing"])["C"]
model = buat_pipeline(C_TERPILIH, char=False).fit(df["teks"], df["label"])
print("C terpilih:", C_TERPILIH)

## Evaluasi 2: soal di game (tidak pernah ikut dilatih)
`buat_dataset.py` juga membuang contoh latih yang mirip (Jaccard ≥ 0.6) dengan soal game. Jumlahnya hanya 36, jadi angkanya kasar; fokus pada soal yang salah.

In [ ]:
pred = model.predict(game["teks"])
prob = model.predict_proba(game["teks"])[:, 1]
print("Akurasi:", accuracy_score(game["label"], pred))
print(classification_report(game["label"], pred, target_names=["Normal", "Phishing"], zero_division=0))
print("Confusion matrix [normal, phishing]:\n", confusion_matrix(game["label"], pred))
game.assign(prob_phishing=prob.round(2), prediksi=pred)[["id", "label", "prediksi", "prob_phishing", "teks"]]

## Evaluasi 3: pesan tulisan tangan di luar template
`data/uji_mandiri.csv` (78 pesan) ditulis tangan, bukan hasil template. `data/uji_buta.csv` (40 pesan) ditulis belakangan dan dipakai untuk membandingkan model lama vs baru. Keduanya tidak ikut dilatih, jadi gambarannya lebih dekat ke dunia nyata. Tetap ingat: ukurannya kecil dan ditulis oleh pembuat sistem, jadi angka ini bukan jaminan.


In [ ]:
for nama in ("uji_mandiri", "uji_buta"):
    u = pd.read_csv(f"data/{nama}.csv")
    pr = model.predict(u["teks"]); pb = model.predict_proba(u["teks"])[:, 1]
    print(nama, "akurasi:", round(accuracy_score(u["label"], pr), 3))
    print(classification_report(u["label"], pr, target_names=["Normal", "Phishing"], zero_division=0))
    salah = u.assign(prob=pb.round(2), pred=pr); print(salah[salah["label"] != salah["pred"]][["label", "prob", "teks"]].to_string(), "\n")


## Ciri apa yang paling berpengaruh?
Koefisien positif mendorong prediksi ke *phishing*, negatif ke *normal*. Token berawalan `z` adalah ciri tautan dan pola penting.

In [ ]:
tf = model.named_steps["tfidf"]; clf = model.named_steps["clf"]
nama = np.array(tf.get_feature_names_out()); koef = clf.coef_[0]
urut = np.argsort(koef)
print("Mendorong NORMAL :", [n.split("__", 1)[-1] for n in nama[urut[:15]]])
print("Mendorong PHISHING:", [n.split("__", 1)[-1] for n in nama[urut[-15:][::-1]]])

## Uji coba kalimat bebas (mirip tab Tweet di game)

In [ ]:
uji = [
    "Klik bit.ly/promo-gratis-88 isi data kartu kamu buat klaim hadiah",
    "Jangan pernah kasih tau OTP ke siapa pun ya guys, banyak penipuan",
    "Info lomba debat, daftar di https://forms.gle/abc123 sampai Jumat",
    "kak transfer dulu 200rb ke rek 1234567890 darurat",
    "Yuk nonton bareng sabtu ini, aku yang beliin popcorn",
]
for t, p in zip(uji, model.predict_proba(uji)[:, 1]):
    print(f"{p:5.2f}  {t}")

## Simpan model untuk server
File ini dibaca `server.py`. Jika versi scikit-learn di laptop berbeda dari server dan muncul error saat memuat, jalankan ulang sel ini (atau `python train_medsos.py`) di lingkungan yang sama dengan server.

In [ ]:
import joblib, json, sklearn
joblib.dump(model, "model_medsos.joblib", compress=3)
json.dump({"sklearn": sklearn.__version__, "n_latih": int(len(df))}, open("model_meta.json", "w"))
print("Tersimpan: model_medsos.joblib (scikit-learn", sklearn.__version__ + ")")


## Membandingkan dengan model lama (opsional, untuk laporan)
Di `Data_Prep.ipynb`, setelah model lama dilatih, tambahkan sel ini dan salin `data/game_items.csv` ke sana:
```python
g = pd.read_csv("game_items.csv")
X = tfidf.transform(g["teks"].map(bersihkan_teks).map(preprocess))
print(accuracy_score(g["label"], model.predict(X)))
```
Bandingkan angkanya dengan akurasi di atas. Itu bukti langsung bahwa model baru lebih cocok dengan domain game.